In [4]:
from pathlib import Path
import tarfile
import urllib
import numpy as np
import pandas as pd

In [5]:
def fetch_data():
    BASE_URL = "https://spamassassin.apache.org/old/publiccorpus/"
    HAM_URL = BASE_URL + "20030228_easy_ham.tar.bz2"
    SPAM_URL = BASE_URL + "20030228_spam.tar.bz2"
    
    DATA_DIR = Path() / "data"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    
    for url, tar in ((HAM_URL, "easy_ham"),
                            (SPAM_URL, "spam")):
        if not (DATA_DIR / tar).is_dir():
            tar_path = (DATA_DIR / tar).with_suffix(".tar.bz2")
            urllib.request.urlretrieve(url, tar_path)
            
            tar_bz2 = tarfile.open(tar_path)
            tar_bz2.extractall(path=DATA_DIR)
    return [(DATA_DIR / dir) for dir in ("easy_ham", "spam")]

In [6]:
ham_dir, spam_dir = fetch_data()

In [7]:
ham_files = [f for f in sorted(ham_dir.iterdir()) if len(f.name) > 20]
spam_files = [f for f in sorted(spam_dir.iterdir()) if len(f.name) > 20]

In [8]:
len(ham_files), len(spam_files), spam_files[0]

(2500, 500, PosixPath('data/spam/00001.7848dde101aa985090474a91ec93fcf0'))

In [9]:
import email
import email.policy

def read_email(path):
    with open(path, "rb") as f:
        return email.parser.BytesParser(policy=email.policy.default).parse(f)

In [10]:
print(read_email(spam_files[0]))

Return-Path: <12a1mailbot1@web.de>
Delivered-To: zzzz@localhost.spamassassin.taint.org
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.spamassassin.taint.org (Postfix) with ESMTP id 136B943C32
	for <zzzz@localhost>; Thu, 22 Aug 2002 08:17:21 -0400 (EDT)
Received: from mail.webnote.net [193.120.211.219]
	by localhost with POP3 (fetchmail-5.9.0)
	for zzzz@localhost (single-drop); Thu, 22 Aug 2002 13:17:21 +0100 (IST)
Received: from dd_it7 ([210.97.77.167])
	by webnote.net (8.9.3/8.9.3) with ESMTP id NAA04623
	for <zzzz@spamassassin.taint.org>; Thu, 22 Aug 2002 13:09:41 +0100
From: 12a1mailbot1@web.de
Received: from r-smtp.korea.com - 203.122.2.197 by dd_it7  with Microsoft
 SMTPSVC(5.5.1775.675.6);	 Sat, 24 Aug 2002 09:42:10 +0900
To: <dcek1a1@netsgo.com>
Subject: Life Insurance - Why Pay More?
Date: Wed, 21 Aug 2002 20:31:57 -1600
MIME-Version: 1.0
Message-ID: <0103c1042001882DD_IT7@dd_it7>
Content-Type: text/html; charset="iso-8859-1"
Content-Transfer-Encoding: quoted-

In [11]:
ham_mails = [read_email(m) for m in ham_files]
spam_mails = [read_email(m) for m in spam_files]

In [12]:
print(ham_mails[0].get_content().strip())

Date:        Wed, 21 Aug 2002 10:54:46 -0500
    From:        Chris Garrigues <cwg-dated-1030377287.06fa6d@DeepEddy.Com>
    Message-ID:  <1029945287.4797.TMDA@deepeddy.vircio.com>


  | I can't reproduce this error.

For me it is very repeatable... (like every time, without fail).

This is the debug log of the pick happening ...

18:19:03 Pick_It {exec pick +inbox -list -lbrace -lbrace -subject ftp -rbrace -rbrace} {4852-4852 -sequence mercury}
18:19:03 exec pick +inbox -list -lbrace -lbrace -subject ftp -rbrace -rbrace 4852-4852 -sequence mercury
18:19:04 Ftoc_PickMsgs {{1 hit}}
18:19:04 Marking 1 hits
18:19:04 tkerror: syntax error in expression "int ...

Note, if I run the pick command by hand ...

delta$ pick +inbox -list -lbrace -lbrace -subject ftp -rbrace -rbrace  4852-4852 -sequence mercury
1 hit

That's where the "1 hit" comes from (obviously).  The version of nmh I'm
using is ...

delta$ pick -version
pick -- nmh-1.0.4 [compiled on fuchsia.cs.mu.OZ.AU at Sun Mar 17 14:55:56 

In [13]:
# Weird data structure so probably some emails are not just str
print(spam_mails[0].get_content().strip())

<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN">
<HTML><HEAD>
<META content="text/html; charset=windows-1252" http-equiv=Content-Type>
<META content="MSHTML 5.00.2314.1000" name=GENERATOR></HEAD>
<BODY><!-- Inserted by Calypso -->
<TABLE border=0 cellPadding=0 cellSpacing=2 id=_CalyPrintHeader_ rules=none 
style="COLOR: black; DISPLAY: none" width="100%">
  <TBODY>
  <TR>
    <TD colSpan=3>
      <HR color=black noShade SIZE=1>
    </TD></TR></TD></TR>
  <TR>
    <TD colSpan=3>
      <HR color=black noShade SIZE=1>
    </TD></TR></TBODY></TABLE><!-- End Calypso --><!-- Inserted by Calypso --><FONT 
color=#000000 face=VERDANA,ARIAL,HELVETICA size=-2><BR></FONT></TD></TR></TABLE><!-- End Calypso --><FONT color=#ff0000 
face="Copperplate Gothic Bold" size=5 PTSIZE="10">
<CENTER>Save up to 70% on Life Insurance.</CENTER></FONT><FONT color=#ff0000 
face="Copperplate Gothic Bold" size=5 PTSIZE="10">
<CENTER>Why Spend More Than You Have To?
<CENTER><FONT color=#ff0000 face="Copp

In [14]:
# So we can get quite a lot of info from the e-mails not just text, but first let's split it into train / test

In [15]:
from collections import Counter

In [16]:
def get_mail_structure(mail):
    payload = mail.get_payload()
    if isinstance(payload, list):
        multipart = ', '.join([get_mail_structure(sub_mail)
                               for sub_mail in payload])
        return f"multipart({multipart})"
    else:
        return mail.get_content_type()

In [17]:
from collections import Counter

In [19]:
def count_structures(mails):
    structures = Counter()
    for mail in mails:
        structure = get_mail_structure(mail)
        structures[structure] += 1
    return structures

In [121]:
from sklearn.model_selection import train_test_split

X = np.array(ham_mails + spam_mails, dtype=object)
y = np.array([0] * len(ham_mails) + [1] * len(spam_mails))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

In [122]:
# We need to get text data
## if html need to convert to plain text
# remove punctuation
# catch URLs
# catch numbers
# perform stemming

In [276]:
import string
import re
from bs4 import BeautifulSoup

def html_to_text(html_data):
    gfg = BeautifulSoup(html_data)
    return gfg.get_text()

# Simple helper functions
def get_mail_text(mail, strip_headers=False):
    for part in mail.walk():
        type = part.get_content_type()
        if type not in ("text/plain", "text/html"):
            continue
        try:
            content = part.get_content()
        except:
            content = str(part.get_payload())
        if type == "text/plain":
           return content
        else:
            headers = None
            if not strip_headers:
                headers = mail.items()
            return str(headers) + html_to_text(content)

def remove_urls(text):
    URL_PATTERN = "(https?://\S+|www\.\S+)"
    text = re.sub(rf'{URL_PATTERN}', "URL", text)
    return text 

def remove_numbers(text):
    text = re.sub(r'\d+', "NUMBER", text)
    return text

def remove_punctuation(text):
    text = text.translate(str.maketrans('', '', string.punctuation))
    return " ".join(text.split())

<>:28: SyntaxWarning: invalid escape sequence '\S'
<>:28: SyntaxWarning: invalid escape sequence '\S'
/var/folders/sm/r88tc34s4cb2nhb0m6308ty40000gn/T/ipykernel_81153/2299521470.py:28: SyntaxWarning: invalid escape sequence '\S'
  URL_PATTERN = "(https?://\S+|www\.\S+)"


In [277]:
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import nltk 
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/jakubwalkowicz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class EmailToWordCountTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, strip_headers=False, lower_case=True, remove_punctuation=True, replace_url=True, replace_nums=True, stemming=True):
        self.strip_headers = strip_headers
        self.lower_case = lower_case
        self.remove_punctuation = remove_punctuation
        self.replace_url = replace_url
        self.replace_nums = replace_nums
        self.stemming = stemming
    
    def fit(self, X, y=None):
        if self.stemming:
            self.stemmer = PorterStemmer()
        return self
            
    def transform(self, X):
        X = [self._clean_mail(get_mail_text(mail, self.strip_headers) or "") for mail in X]
        return np.array([Counter(row) for row in X])
        
    def get_feature_names_out(self, input_features=None):
        return np.array(["Text"])
    
    def _clean_mail(self, mail):
        if self.lower_case: 
            mail = mail.lower()
        if self.replace_nums:
            mail = remove_numbers(mail)
        if self.replace_url:
            mail = remove_urls(mail)
        if self.remove_punctuation:
            mail = remove_punctuation(mail)
        if self.stemming and self.stemmer is not None:
            return self._get_stemming(mail)
        return mail.split()
    
    def _get_stemming(self, text):
        tokens = word_tokenize(text)
        stemmed = [self.stemmer.stem(token) for token in tokens if len(token) >= 2]
        return stemmed

In [401]:
from scipy.sparse import csr_matrix

class WordCountToSparseVectorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=1000):
        self.max_features = max_features
    
    def fit(self, X, y=None):
        dataset_counter = Counter()
        for row in X:
            dataset_counter.update(row)
        most_common = dataset_counter.most_common(self.max_features)        
        self.vocabulary_ = {word: index + 1 for index, (word, count) in enumerate(most_common)}
        return self
    
    def transform(self, X):
        data = []
        row = []
        col = []
        for index, counter in enumerate(X):
            for word in counter:
                data.append(counter[word])
                row.append(index)
                col.append(self.vocabulary_.get(word, 0))
        return csr_matrix((data, (row, col)), shape=(len(X), self.max_features + 1))

In [402]:
from sklearn.pipeline import make_pipeline

preprocessing = make_pipeline(
    EmailToWordCountTransformer(),
    WordCountToSparseVectorTransformer(),
)

In [409]:
X_train_transformed = preprocessing.fit_transform(X_train)

In [410]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

log_clf = LogisticRegression(random_state=42)
score = cross_val_score(log_clf, X_train_transformed, y_train, cv=3)

/Users/jakubwalkowicz/danger-ai-zone/hands_on_ml_book/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/jakubwalkowicz/danger-ai-zone/hands_on_ml_book/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the dat

In [411]:
score.mean()

np.float64(0.98875)

In [413]:
from sklearn.metrics import precision_score, recall_score

In [414]:
X_test_transformed = preprocessing.transform(X_test)
log_clf = LogisticRegression(max_iter=1000, random_state=42)
log_clf.fit(X_train_transformed, y_train)
y_pred = log_clf.predict(X_test_transformed)

print(f"Precision: {precision_score(y_test, y_pred):.2%}")
print(f"Recall: {recall_score(y_test, y_pred):.2%}")

Precision: 93.81%
Recall: 95.79%
